# CBOE SPX Options Chain Ingestion and Gamma Exposure Calculation
A test of sourcing S&P 500 options chain data from CBOE and calulating the gamma exposure. This is based on a whole host of assumptions, which I think are actually quite reasonable.

The distance to the gamma flip line is highly predictive of future volatility spikes.

**Author:** puzzle

**Created:** 2026-01-22

**Modified:** 2026-03-12


## Import Modules


In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import requests
from scipy.stats import norm

In [2]:
from jfmi.plot.utilities import load_plotly_templates
from jfmi.utilities.calendar import process_interactive_brokers_trading_hours

In [3]:
from jfmi.plot.templates import COLOURS

In [4]:
load_plotly_templates()

### Ingest Options Chain


In [5]:
def calculate_gamma_exposure(S, K, vol, T, r, q, option_type, open_interest):
    """Calculate Black-Scholes European-Options Gamma."""
    if T == 0 or vol == 0:
        return 0

    dp = (np.log(S / K) + (r - q + 0.5 * vol**2) * T) / (vol * np.sqrt(T))
    dm = dp - vol * np.sqrt(T)

    # Gamma is the same for both calls and puts (this is just to cross-check).
    if option_type == "call":
        gamma = np.exp(-q * T) * norm.pdf(dp) / (S * vol * np.sqrt(T))
        return open_interest * 100 * S * S * 0.01 * gamma
    else:
        gamma = K * np.exp(-r * T) * norm.pdf(dm) / (S * S * vol * np.sqrt(T))
        return open_interest * 100 * S * S * 0.01 * gamma

In [6]:
index = "SPX"

response = requests.get(
    url="https://cdn.cboe.com/api/global/delayed_quotes/options/_" + index + ".json",
    timeout=60,
)
data = response.json()

In [7]:
spot_price = data["data"]["close"]

from_strike = round(0.8 * spot_price / 50) * 50
to_strike = round(1.2 * spot_price / 50) * 50

date_today = pd.Timestamp.today().date()

In [8]:
df_data = pd.DataFrame(data["data"]["options"])

df_data["type"] = df_data["option"].str.slice(start=-9, stop=-8)

df_data["expiration_date"] = df_data["option"].str.slice(start=-15, stop=-9)
df_data["expiration_date"] = pd.to_datetime(df_data["expiration_date"], format="%y%m%d")

df_data["strike_price"] = df_data["option"].str.slice(start=-8, stop=-3)
df_data["strike_price"] = df_data["strike_price"].str.lstrip("0")

In [9]:
df_data_calls = df_data.loc[df_data["type"] == "C"].reset_index(drop=True)
df_data_puts = df_data.loc[df_data["type"] == "P"].reset_index(drop=True)

columns = df_data.columns[
    ~df_data.columns.isin(
        [
            "change",
            "percent_change",
            "open",
            "high",
            "low",
            "prev_day_close",
            "tick",
            "type",
        ]
    )
]

df_calls = df_data_calls[columns].add_prefix("call_")
df_puts = df_data_puts[columns].add_prefix("put_")

In [10]:
df_options = pd.concat([df_calls, df_puts], axis=1)

df_options.drop(["put_expiration_date", "put_strike_price"], axis=1, inplace=True)
df_options.rename(
    {
        "call_iv": "call_implied_volatility",
        "put_iv": "put_implied_volatility",
        "call_strike_price": "strike_price",
        "call_expiration_date": "expiration_date",
    },
    axis=1,
    inplace=True,
)

df_options["strike_price"] = df_options["strike_price"].astype(float)

df_options["expiration_date"] = pd.to_datetime(df_options["expiration_date"])
df_options["call_last_trade_time"] = pd.to_datetime(df_options["put_last_trade_time"])
df_options["put_last_trade_time"] = pd.to_datetime(df_options["put_last_trade_time"])

df_options["expiration_date"] = df_options["expiration_date"] + pd.Timedelta(hours=16)

### Calculate Spot Gamma

To further convert into 'per 1% move', multiply by 1% of the spot price.


In [11]:
df_options["call_gamma_exposure"] = (
    df_options["call_gamma"]  # unit gamma
    * df_options["call_open_interest"]
    * 100  # contract size
    * spot_price**2
    * 0.01
)
df_options["put_gamma_exposure"] = (
    df_options["put_gamma"]  # unit gamma
    * df_options["put_open_interest"]
    * 100  # contract size
    * spot_price**2
    * 0.01
    * -1
)

df_options["total_gamma"] = (
    df_options["call_gamma_exposure"] + df_options["put_gamma_exposure"]
) / 1e9  # billion

In [12]:
columns_groupby = ["strike_price"]

dict_groupby = {
    column: "sum" for column in df_options.columns if column not in columns_groupby
}

dict_groupby["expiration_date"] = "last"
dict_groupby["call_last_trade_time"] = "last"
dict_groupby["put_last_trade_time"] = "last"

df_options_by_strike_price = df_options.groupby(columns_groupby).agg(dict_groupby)

strikes = df_options_by_strike_price.index.values

### Plot Absolute Gamma Exposure


In [13]:
template = pio.templates[pio.templates.default]

colour_map = {
    "increasing": template.data.candlestick[0].increasing.line.color,
    "decreasing": template.data.candlestick[0].decreasing.line.color,
}

In [14]:
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=strikes,
        y=df_options_by_strike_price["total_gamma"],
        name="Total Gamma Exposure",
        marker=dict(
            # color=COLOURS["blue"][4],
            color=np.where(
                df_options_by_strike_price["total_gamma"] >= 0,
                colour_map["increasing"],
                colour_map["decreasing"],
            ),
            line=dict(width=0),
        ),
    )
)

# Plot the spot price.
fig.add_vline(
    x=spot_price,
    line=dict(color=COLOURS["red"][4], width=2),
    annotation_text=f" {index} Spot: {spot_price:,.2f}",  # space for position
    annotation_position="top right",
)

fig.update_layout(
    title=f"Total Gamma: ${df_options['total_gamma'].sum():.2f} Bn per 1% {index} Move",
    xaxis=dict(
        title="Strike Price",
        range=[from_strike, to_strike],
    ),
    yaxis=dict(title="Spot Gamma Exposure ($ billions / 1% move)"),
    margin=dict(t=50),
)

fig.show()

### Plot Absolute Gamma Exposure by Calls and Puts


In [15]:
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=strikes,
        y=df_options_by_strike_price["call_gamma_exposure"] / 1e9,
        name="Call Gamma Exposure",
        marker=dict(
            color=colour_map["increasing"],
            line=dict(width=0),
        ),
    )
)

fig.add_trace(
    go.Bar(
        x=strikes,
        y=df_options_by_strike_price["put_gamma_exposure"] / 1e9,
        name="Put Gamma Exposure",
        marker=dict(
            color=colour_map["decreasing"],
            line=dict(width=0),
        ),
    )
)

# Plot the spot price.
fig.add_vline(
    x=spot_price,
    line=dict(color=COLOURS["red"][4], width=2),
    annotation_text=f" {index} Spot: {spot_price:,.2f}",  # space for position
    annotation_position="top right",
)

fig.update_layout(
    title=f"Total Gamma: ${df_options['total_gamma'].sum():.2f} Bn per 1% {index} Move",
    xaxis=dict(
        title="Strike Price",
        range=[from_strike, to_strike],
    ),
    yaxis=dict(title="Spot Gamma Exposure ($ billions / 1% move)"),
    margin=dict(t=80),
    barmode="relative",
)

fig.show()

### Plot Open Interest by Calls and Puts


In [16]:
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=strikes,
        y=df_options_by_strike_price["call_open_interest"],
        name="Call Open Interest",
        marker=dict(
            color=colour_map["increasing"],
            line=dict(width=0),
        ),
    )
)

fig.add_trace(
    go.Bar(
        x=strikes,
        y=-1 * df_options_by_strike_price["put_open_interest"],
        name="Put Open Interest",
        marker=dict(
            color=colour_map["decreasing"],
            line=dict(width=0),
        ),
    )
)

# Plot the spot price.
fig.add_vline(
    x=spot_price,
    line=dict(color=COLOURS["red"][4], width=2),
    annotation_text=f" {index} Spot: {spot_price:,.2f}",  # space for position
    annotation_position="top right",
)

fig.update_layout(
    xaxis=dict(
        title="Strike Price",
        range=[from_strike, to_strike],
    ),
    yaxis=dict(title="Open Interest (number of contracts)"),
    # Put calls above the x-axis and puts below.
    barmode="relative",
)

fig.show()

### Calculate Gamma Exposure Profile


In [17]:
levels = np.arange(from_strike, to_strike + 50, 50)

# Set DTE to 1 day for 0DTE options so that they're not excluded.
df_options["days_until_expiry"] = [
    1 / 262
    if (np.busday_count(date_today, x.date())) == 0
    else np.busday_count(date_today, x.date()) / 262
    for x in df_options.expiration_date
]

date_next_expiry = df_options["expiration_date"].min()

df_options["is_third_friday"] = [
    x.weekday() == 4 and 15 <= x.day <= 21 for x in df_options.expiration_date
]
third_fridays = df_options.loc[df_options["is_third_friday"]]

date_next_monthy_expiry = third_fridays["expiration_date"].min()

In [18]:
# Calculate the gamma exposure is at each level.
total_gamma = []
total_gamma_excluding_next = []
total_gamma_excluding_friday = []

for level in levels:
    df_options["call_gamma_exposure"] = df_options.apply(
        lambda row: calculate_gamma_exposure(
            level,
            row["strike_price"],
            row["call_implied_volatility"],
            row["days_until_expiry"],
            0,
            0,
            "call",
            row["call_open_interest"],
        ),
        axis=1,
    )

    df_options["put_gamma_exposure"] = df_options.apply(
        lambda row: calculate_gamma_exposure(
            level,
            row["strike_price"],
            row["put_implied_volatility"],
            row["days_until_expiry"],
            0,
            0,
            "put",
            row["put_open_interest"],
        ),
        axis=1,
    )

    total_gamma.append(
        df_options["call_gamma_exposure"].sum() - df_options["put_gamma_exposure"].sum()
    )

    gamma_excluding_next = df_options.loc[
        df_options["expiration_date"] != date_next_expiry
    ]
    total_gamma_excluding_next.append(
        gamma_excluding_next["call_gamma_exposure"].sum()
        - gamma_excluding_next["put_gamma_exposure"].sum()
    )

    gamma_excluding_friday = df_options.loc[
        df_options["expiration_date"] != date_next_monthy_expiry
    ]
    total_gamma_excluding_friday.append(
        gamma_excluding_friday["call_gamma_exposure"].sum()
        - gamma_excluding_friday["put_gamma_exposure"].sum()
    )

total_gamma = np.array(total_gamma) / 1e9
total_gamma_excluding_next = np.array(total_gamma_excluding_next) / 1e9
total_gamma_excluding_friday = np.array(total_gamma_excluding_friday) / 1e9

In [19]:
# Find the gamma flip point.
idx_flip = np.where(np.diff(np.sign(total_gamma)))[0]

negative_gamma = total_gamma[idx_flip]
positive_gamma = total_gamma[idx_flip + 1]

negative_strike = levels[idx_flip]
positive_strike = levels[idx_flip + 1]

zero_gamma = positive_strike - (
    (positive_strike - negative_strike)
    * positive_gamma
    / (positive_gamma - negative_gamma)
)
zero_gamma = zero_gamma[0]

### Plot Gamma Exposure Profile


In [20]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=levels,
        y=total_gamma,
        mode="lines",
        name="All Expiries",
        line=dict(color=COLOURS["blue"][4]),
    )
)
fig.add_trace(
    go.Scatter(
        x=levels,
        y=total_gamma_excluding_next,
        mode="lines",
        name="Excluding Next Expiry",
        line=dict(color=COLOURS["orange"][4]),
    )
)
fig.add_trace(
    go.Scatter(
        x=levels,
        y=total_gamma_excluding_friday,
        mode="lines",
        name="Excluding Next Monthly Expiry",
        line=dict(color=COLOURS["purple"][4]),
    )
)

# Plot the spot price.
fig.add_vline(
    x=spot_price,
    line=dict(color=COLOURS["red"][4], width=2),
    annotation_text=f" {index} Spot: {spot_price:,.2f}",  # space for position
    annotation_position="top right",
)

# Plot the zero gamma price.
fig.add_vline(
    x=zero_gamma,
    line=dict(color=COLOURS["green"][4], width=2),
    annotation_text=f"{index} Gamma Flip: {zero_gamma:,.2f} ",
    annotation_position="bottom right",
)

fig.add_hline(y=0, line=dict(color=COLOURS["grey"][4], width=2))

# Shade areas of negative and positive gamma.
fig.add_traces(
    [
        go.Scatter(
            x=[from_strike, zero_gamma, zero_gamma, from_strike],
            y=[min(total_gamma), min(total_gamma), max(total_gamma), max(total_gamma)],
            fill="toself",
            fillcolor="rgba(255,0,0,0.1)",  # red
            line=dict(width=0),
            showlegend=False,
        ),
        go.Scatter(
            x=[zero_gamma, to_strike, to_strike, zero_gamma],
            y=[min(total_gamma), min(total_gamma), max(total_gamma), max(total_gamma)],
            fill="toself",
            fillcolor="rgba(0,255,0,0.1)",  # green
            line=dict(width=0),
            showlegend=False,
        ),
    ]
)

fig.update_layout(
    title=f"Gamma Exposure Profile, {index}, {date_today.strftime('%d %b %Y')}",
    xaxis=dict(title="Index Price", range=[from_strike, to_strike]),
    yaxis=dict(title="Gamma Exposure ($ billions/1% move)"),
    margin=dict(t=80),
)

fig.show()